# ADM1 Benchmark — Custom Controller Tutorial

This notebook covers three workflows:

1. **Write and test a rule-based controller** — implement the `BaseController` interface, run it on any scenario, read the metrics
2. **Train an RL agent (SAC)** — configure reward, train for N timesteps, view the learning curve
3. **Compare controllers** — run both through the full benchmark and compare results side-by-side

**Prerequisites** — install the package from the repo root:
```bash
pip install -e .
```

In [ ]:
import sys
from pathlib import Path

# Point to repo root so all imports work
REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 110})

# Core components
from env.adm1_gym_env import ADM1Env_v2
from evaluation.metrics_calculator import MetricsCalculator
from baselines.baseline_controllers import BaseController

print("Imports OK")

---
## Part 1 — Environment Overview

### Observation space (13-dim, `obs_mode='full'`)

| idx | variable | unit | description |
|-----|----------|------|-------------|
| 0 | `total_vfa` | kg COD/m³ | total volatile fatty acids |
| 1 | `alkalinity` | kmol/m³ | buffering capacity |
| 2 | `vfa_alk_ratio` | — | VFA / alkalinity |
| 3 | `S_h2` | kg COD/m³ | dissolved hydrogen |
| **4** | **`pH`** | — | **reactor pH** |
| 5 | `S_nh3` | kmol N/m³ | free ammonia |
| 6 | `S_IN` | kmol N/m³ | inorganic nitrogen |
| 7 | `X_ac` | kg COD/m³ | acetoclastic biomass |
| 8 | `X_h2` | kg COD/m³ | hydrogenotrophic biomass |
| **9** | **`q_ch4`** | m³/day | **methane flow rate** |
| 10 | `q_ad_current` | m³/day | feed flow (previous step) |
| 11 | `feed_mult_current` | — | feed multiplier (previous step) |
| 12 | `T_L_norm` | — | (T_liquid − 35 °C) / 10 |

Compact mode (`obs_mode='simple'`) gives indices `[4, 9, 12, 10, 11]` only.

### Action space (3-dim continuous, **fixed bounds**)

| dim | variable | unit | range |
|-----|----------|------|-------|
| 0 | `q_ad` | m³/day | **[50, 300]** |
| 1 | `feed_mult` | — | **[0.7, 1.3]** |
| 2 | `Q_HEX` | W | **[−5000, 5000]** |

> **The action bounds are hard constraints** — values outside are clipped by the environment.  
> The observation space is what the simulator reports; your controller can **read any subset** of it.

### Safety thresholds (used for scoring)

| variable | violation if | termination if |
|----------|-------------|----------------|
| pH | < 6.8 or > 7.8 | < 5.8 |
| total_vfa | > 0.2 kg COD/m³ | > 0.8 |
| S_nh3 | > 0.002 kmol N/m³ | > 0.01 |

In [ ]:
# Inspect the environment directly
env = ADM1Env_v2(scenario_name='nominal', obs_mode='full')
obs, _ = env.reset(seed=42)

print(f"Observation shape : {obs.shape}")
print(f"Action space      : {env.action_space}")
print()
print("Initial observation:")
labels = [
    "total_vfa", "alkalinity", "vfa_alk_ratio", "S_h2",
    "pH", "S_nh3", "S_IN", "X_ac", "X_h2",
    "q_ch4", "q_ad_current", "feed_mult_current", "T_L_norm",
]
for i, (lbl, val) in enumerate(zip(labels, obs)):
    print(f"  obs[{i:2d}]  {lbl:<22} = {val:.4f}")

env.close()

---
## Part 2 — Implement Your Controller

Subclass `BaseController` and implement two methods:
- `reset()` — called once per episode (clear integrators, history, etc.)
- `get_action(obs)` — called every 15-minute step; returns `[q_ad, feed_mult, Q_HEX]`

Your controller can use **any subset** of the 13 observations.  
Actions outside the bounds `[50,300] × [0.7,1.3] × [-5000,5000]` are silently clipped.

In [ ]:
# ── Observation index shortcuts ───────────────────────────────────────────────
OBS_VFA     = 0   # total VFA  (kg COD/m³)
OBS_PH      = 4   # pH
OBS_Q_CH4   = 9   # methane flow (m³/day)
OBS_Q_AD    = 10  # feed flow, previous step
OBS_FEED    = 11  # feed multiplier, previous step
OBS_T_NORM  = 12  # (T_liquid − 35 °C) / 10


class MyController(BaseController):
    """
    Example: dead-band pH controller with thermal management.

    Feel free to replace this with any algorithm — MPC, PID, rule-based, etc.
    The only contract is get_action(obs) → length-3 array.
    """

    def __init__(self, ph_target=7.2, kp=0.20):
        super().__init__(name="MyController")
        self.ph_target   = ph_target
        self.ph_deadband = 0.15
        self.kp          = kp           # proportional gain: ΔpH → Δfeed_mult
        self._integral   = 0.0         # optional: add integral term

    def reset(self):
        self.step_count = 0
        self._integral  = 0.0

    def get_action(self, obs: np.ndarray) -> np.ndarray:
        self.step_count += 1

        ph        = obs[OBS_PH]
        total_vfa = obs[OBS_VFA]
        T_norm    = obs[OBS_T_NORM] if len(obs) > 5 else 0.0

        # ── Feed rate: fixed nominal ──────────────────────────────────────
        q_ad = 180.0

        # ── Feed multiplier: P controller on pH ───────────────────────────
        ph_err = self.ph_target - ph
        if abs(ph_err) <= self.ph_deadband:
            feed_mult = 1.0
        else:
            feed_mult = 1.0 - self.kp * ph_err

        # VFA safety clamp
        if total_vfa > 0.15:
            feed_mult = min(feed_mult, 0.85)

        # ── Heating: proportional to temperature deficit ──────────────────
        T_liq = T_norm * 10.0 + 35.0       # approximate °C
        if T_liq < 33.0:
            q_hex = 3000.0
        elif T_liq > 38.0:
            q_hex = -500.0
        else:
            q_hex = 0.0

        return np.array([
            np.clip(q_ad,      50.0,   300.0),
            np.clip(feed_mult,  0.7,     1.3),
            np.clip(q_hex,   -5000.0, 5000.0),
        ], dtype=np.float32)


print("MyController defined.")

---
## Part 3 — Run One Episode and Visualise

A helper that runs the controller for one episode and collects time-series data.

In [ ]:
def run_episode(controller, scenario='nominal', obs_mode='full', seed=42):
    """Run one episode, return metrics dict + time-series arrays."""
    env  = ADM1Env_v2(scenario_name=scenario, obs_mode=obs_mode)
    obs, _ = env.reset(seed=seed)
    controller.reset()
    calc = MetricsCalculator()

    ts = {"ph": [], "vfa": [], "ch4": [], "feed_mult": [], "q_hex": [], "T_liq": []}
    done, step = False, 0

    while not done:
        action = controller.get_action(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        calc.add_step(obs, action, reward, info)
        step += 1
        if terminated:
            calc.set_terminated(step)
        done = terminated or truncated

        ts["ph"].append(obs[4])
        ts["vfa"].append(obs[0])
        ts["ch4"].append(obs[9])
        ts["feed_mult"].append(action[1])
        ts["q_hex"].append(action[2])
        ts["T_liq"].append(obs[12] * 10.0 + 35.0 if len(obs) > 5 else float("nan"))

    env.close()
    metrics = calc.compute_metrics()
    return metrics, {k: np.array(v) for k, v in ts.items()}

# ── Run ───────────────────────────────────────────────────────────────────────
ctrl = MyController(ph_target=7.2, kp=0.20)
metrics, ts = run_episode(ctrl, scenario='nominal')

m = metrics["summary"]
s = metrics["safety"]
p = metrics["production"]
print(f"Overall score  : {m['overall_score']:.3f}")
print(f"Violation rate : {s['violation_rate']*100:.1f}%")
print(f"Avg CH4        : {p['avg_ch4_flow']:.1f} m³/day")
print(f"Mean pH        : {s['ph_mean']:.3f}")
print(f"Max VFA        : {s['vfa_max']:.4f} kg COD/m³")

In [ ]:
t_days = np.arange(len(ts["ph"])) * (15 / 1440)   # 15-min steps → days

fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
fig.suptitle("Episode trajectory — MyController on Nominal", fontweight="bold")

def _plot(ax, y, ylabel, color, hlines=None):
    ax.plot(t_days, y, color=color, lw=1.2)
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", ls=":", lw=0.5)
    if hlines:
        for val, ls, label in hlines:
            ax.axhline(val, ls=ls, lw=0.9, color="crimson", label=label)
        ax.legend(fontsize=8)

_plot(axes[0, 0], ts["ph"],        "pH",            "#0072B2",
      [(6.8, "--", "safe min"), (7.8, "--", "safe max")])
_plot(axes[0, 1], ts["vfa"],       "VFA (kg/m³)",   "#D55E00",
      [(0.2, "--", "viol thresh")])
_plot(axes[1, 0], ts["ch4"],       "CH4 (m³/day)",  "#009E73")
_plot(axes[1, 1], ts["feed_mult"], "feed_mult",      "#CC79A7",
      [(0.7, ":", "min"), (1.3, ":", "max")])
_plot(axes[2, 0], ts["q_hex"],     "Q_HEX (W)",     "#56B4E9")
_plot(axes[2, 1], ts["T_liq"],     "T_liq (°C)",    "#E69F00")

for ax in axes[2]:
    ax.set_xlabel("Time (days)")

plt.tight_layout()
plt.show()

---
## Part 4 — Full Benchmark (All 6 Scenarios)

Run your controller on all six paper scenarios and report the same metrics table as the paper.

In [ ]:
SCENARIOS = [
    'nominal', 'high_load', 'low_load',
    'shock_load', 'temperature_drop', 'cold_winter',
]
N_EPISODES = 5   # increase to 10 for paper-quality numbers

def benchmark(controller, n_episodes=N_EPISODES):
    results = {}
    for sc in SCENARIOS:
        scores, viols, ch4s, terms = [], [], [], []
        env = ADM1Env_v2(scenario_name=sc)
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=42 + ep * 100)
            controller.reset()
            calc = MetricsCalculator()
            done, step = False, 0
            while not done:
                action = controller.get_action(obs)
                obs, reward, terminated, truncated, info = env.step(action)
                calc.add_step(obs, action, reward, info)
                step += 1
                if terminated:
                    calc.set_terminated(step)
                done = terminated or truncated
            m = calc.compute_metrics()
            scores.append(m["summary"]["overall_score"])
            viols.append(m["safety"]["violation_rate"])
            ch4s.append(m["production"]["avg_ch4_flow"])
            terms.append(float(m["episode_info"]["terminated_early"]))
        env.close()
        results[sc] = {
            "score": np.mean(scores), "score_std": np.std(scores, ddof=1),
            "viol":  np.mean(viols),  "ch4": np.mean(ch4s),
            "term":  np.mean(terms),
        }
    return results

ctrl = MyController()
res  = benchmark(ctrl)

print(f"\n{'Scenario':<20} {'Score':>7} {'±':>5} {'Viol%':>7} {'CH4':>8}")
print("─" * 52)
for sc, r in res.items():
    print(f"{sc:<20} {r['score']:>7.3f} {r['score_std']:>5.3f} "
          f"{r['viol']*100:>6.1f}%  {r['ch4']:>7.1f}")
print("─" * 52)
print(f"{'Mean':<20} {np.mean([r['score'] for r in res.values()]):>7.3f}")

---
## Part 5 — Train an RL Agent (SAC)

Use the paper's reward configuration (`safety_first`) and train SAC via Stable-Baselines3.  
The cell below trains a lightweight agent for a short run — increase `N_TIMESTEPS` for full training.

> **Tip**: full paper training uses 500,000 timesteps per scenario per seed.  
> A quick experiment with 50,000 timesteps takes ~5 minutes on CPU.

In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from training.reward_configs import REWARD_CONFIGS

# ── Configuration ─────────────────────────────────────────────────────────────
TRAIN_SCENARIO = 'nominal'    # scenario to train on
REWARD_CONFIG  = 'safety_first'  # 'safety_first' | 'sf_linear_only' | 'sf_constant_only'
OBS_MODE       = 'full'          # 'full' (13-dim) | 'simple' (5-dim)
N_TIMESTEPS    = 50_000          # quick demo; paper uses 500_000
SEED           = 42
SAVE_PATH      = REPO / 'results' / 'my_sac_run'

reward_cfg = REWARD_CONFIGS[REWARD_CONFIG]
print(f"Training SAC on '{TRAIN_SCENARIO}' with reward='{REWARD_CONFIG}', "
      f"obs='{OBS_MODE}', steps={N_TIMESTEPS:,}")

# ── Build environments ────────────────────────────────────────────────────────
train_env = Monitor(ADM1Env_v2(
    scenario_name=TRAIN_SCENARIO,
    reward_config=reward_cfg,
    obs_mode=OBS_MODE,
    random_seed=SEED,
))
eval_env = Monitor(ADM1Env_v2(
    scenario_name=TRAIN_SCENARIO,
    reward_config=reward_cfg,
    obs_mode=OBS_MODE,
))

# ── Build SAC model ───────────────────────────────────────────────────────────
model = SAC(
    policy          = 'MlpPolicy',
    env             = train_env,
    learning_rate   = 3e-4,
    buffer_size     = 100_000,
    batch_size      = 256,
    gamma           = 0.99,
    tau             = 0.005,
    ent_coef        = 'auto',
    verbose         = 1,
    seed            = SEED,
    tensorboard_log = str(SAVE_PATH / 'tensorboard'),
)

# ── Train ─────────────────────────────────────────────────────────────────────
callback = EvalCallback(
    eval_env,
    best_model_save_path=str(SAVE_PATH / 'best_model'),
    eval_freq=5_000,
    n_eval_episodes=3,
    deterministic=True,
    verbose=0,
)

model.learn(total_timesteps=N_TIMESTEPS, callback=callback, progress_bar=True)

train_env.close()
eval_env.close()
print(f"\nTraining complete. Best model saved to {SAVE_PATH / 'best_model'}")

---
## Part 6 — Evaluate the Trained SAC Agent

Wrap the SB3 model in a thin `BaseController` adapter so it uses the same benchmark loop.

In [ ]:
class SACController(BaseController):
    """Thin wrapper so a trained SB3 model fits the BaseController interface."""

    def __init__(self, model_or_path, deterministic=True):
        super().__init__(name="SAC")
        if isinstance(model_or_path, (str, Path)):
            self.model = SAC.load(str(model_or_path))
            print(f"Loaded model from {model_or_path}")
        else:
            self.model = model_or_path
        self.deterministic = deterministic

    def reset(self):
        self.step_count = 0

    def get_action(self, obs: np.ndarray) -> np.ndarray:
        self.step_count += 1
        action, _ = self.model.predict(obs, deterministic=self.deterministic)
        return action


# Load best checkpoint saved during training
best_model_path = SAVE_PATH / 'best_model' / 'best_model.zip'
sac_ctrl = SACController(best_model_path)

# Run full benchmark
print("Benchmarking SAC agent ...")
sac_res = benchmark(sac_ctrl)

print(f"\n{'Scenario':<20} {'Score':>7} {'Viol%':>7} {'CH4':>8}")
print("─" * 46)
for sc, r in sac_res.items():
    print(f"{sc:<20} {r['score']:>7.3f} {r['viol']*100:>6.1f}%  {r['ch4']:>7.1f}")
print("─" * 46)
print(f"{'Mean':<20} {np.mean([r['score'] for r in sac_res.values()]):>7.3f}")

---
## Part 7 — Side-by-Side Comparison

Compare your rule-based controller against the trained SAC agent with a bar chart.

In [ ]:
labels = list(SCENARIOS)
x = np.arange(len(labels))
w = 0.35

my_scores  = [res[sc]["score"]    for sc in labels]
sac_scores = [sac_res[sc]["score"] for sc in labels]
my_viols   = [res[sc]["viol"]     for sc in labels]
sac_viols  = [sac_res[sc]["viol"]  for sc in labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.bar(x - w/2, my_scores,  w, label="MyController", color="#0072B2", alpha=0.85)
ax1.bar(x + w/2, sac_scores, w, label="SAC",           color="#E69F00", alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=30, ha="right")
ax1.set_ylabel("Overall score"); ax1.set_title("Overall score (higher = better)")
ax1.axhline(0, color="k", lw=0.7, ls="--"); ax1.legend()

ax2.bar(x - w/2, my_viols,  w, label="MyController", color="#0072B2", alpha=0.85)
ax2.bar(x + w/2, sac_viols, w, label="SAC",           color="#E69F00", alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=30, ha="right")
ax2.set_ylabel("Violation rate"); ax2.set_title("Violation rate (lower = better)")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nMean score   — MyController: {np.mean(my_scores):.3f}  |  SAC: {np.mean(sac_scores):.3f}")
print(f"Mean viol    — MyController: {np.mean(my_viols):.3f}  |  SAC: {np.mean(sac_viols):.3f}")